# Bronze Layer — Insurance Charges
**Purpose:** Land the raw Medical Cost Personal dataset into the lakehouse, untouched.
**Source:** public CSV (1,338 rows, 7 columns)
**Output:** Delta table `bronze_insurance` in lh_medallion
**Rule:** Bronze = faithful copy of source. No cleaning here — that happens in Silver.

#  load the data

In [10]:
import pandas as pd

# Read the insurance data straight from the web into a table called 'pdf'
pdf = pd.read_csv("https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv")

print("Loaded", pdf.shape[0], "rows and", pdf.shape[1], "columns")

StatementMeta(, 845a7a3e-1312-4da9-afae-78c821a5a276, 15, Finished, Available, Finished, False)

Loaded 1338 rows and 7 columns


# EDA

In [11]:
# Show the top 5 rows to get a feel for the data
pdf.head()

StatementMeta(, 845a7a3e-1312-4da9-afae-78c821a5a276, 17, Finished, Available, Finished, False)

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [12]:
# Column names, data types, and how many values are filled in
pdf.info()

StatementMeta(, 845a7a3e-1312-4da9-afae-78c821a5a276, 19, Finished, Available, Finished, False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [13]:
# Count empty cells per column (we want all zeros)
pdf.isnull().sum()

StatementMeta(, 845a7a3e-1312-4da9-afae-78c821a5a276, 20, Finished, Available, Finished, False)

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [14]:
# Quick stats (average, min, max, etc.) for numeric columns
pdf.describe()

StatementMeta(, 845a7a3e-1312-4da9-afae-78c821a5a276, 21, Finished, Available, Finished, False)

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [15]:
# How many of each value in the text columns?
print(pdf["smoker"].value_counts())
print(pdf["region"].value_counts())
print(pdf["sex"].value_counts())

StatementMeta(, 845a7a3e-1312-4da9-afae-78c821a5a276, 22, Finished, Available, Finished, False)

smoker
no     1064
yes     274
Name: count, dtype: int64
region
southeast    364
southwest    325
northwest    325
northeast    324
Name: count, dtype: int64
sex
male      676
female    662
Name: count, dtype: int64


In [16]:
# Average charge: smokers vs non-smokers
pdf.groupby("smoker")["charges"].mean()

StatementMeta(, 845a7a3e-1312-4da9-afae-78c821a5a276, 23, Finished, Available, Finished, False)

smoker
no      8434.268298
yes    32050.231832
Name: charges, dtype: float64

# Write to Bronze (save into the lakehouse)

In [19]:
# === SAVE to lakehouse ===
# Hand the pandas data to Spark, then save it as a permanent Delta table.
df = spark.createDataFrame(pdf)
df.write.mode("overwrite").format("delta").saveAsTable("bronze_insurance")

print("Saved 'bronze_insurance' to lh_medallion.")

StatementMeta(, 845a7a3e-1312-4da9-afae-78c821a5a276, 26, Finished, Available, Finished, False)

Saved 'bronze_insurance' to lh_medallion.


In [20]:
# === VERIFY ===
# Read the table back from the lakehouse to prove it's really there.
check = spark.read.table("bronze_insurance")
print("Rows in lakehouse table:", check.count())   # expect 1338
display(check.limit(5))

StatementMeta(, 845a7a3e-1312-4da9-afae-78c821a5a276, 27, Finished, Available, Finished, False)

Rows in lakehouse table: 1338


SynapseWidget(Synapse.DataFrame, 71aec54d-65e7-4d6d-a22f-e2cbf77e783f)